# One copy, both directions, step by step

A hands-on lab. The Lakebase idea here: your operational rows are the same rows analytics
reads. No change-data-capture job copies data between an OLTP store and an analytics store;
there is one copy, reachable from both sides.

**What runs live here:** a real transaction written through Postgres, a computed fraud-risk score,
written back onto the row, and read back. **What is an explainer (clearly labeled, not run):**
reading the same rows from the lakehouse through a Unity Catalog foreign table on a SQL warehouse.
That needs a metastore-admin catalog registration, so we show the exact query rather than fake it.

**Before you start:** finish `00-setup/`. CTA-100240 is a retail credit account on your branch.

## Setup, connect

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "common", "scripts"))
from lakebase import run_sql
print("ready")

## Step 1, add a fraud_risk column and write a transaction (OLTP)

First make room for an analytics-derived value, then write a new card transaction through
Postgres, a millisecond operational write.

```sql
ALTER TABLE bank.txn ADD COLUMN IF NOT EXISTS fraud_risk NUMERIC(5,1);
INSERT INTO bank.txn (account_id, ts, amount, channel, anomaly_score, mcc, util_pct)
VALUES ('CTA-100240', now(), 38500.00, 'SPEI', 8.7, NULL, 92.0) RETURNING txn_id;
```

In [ ]:
run_sql("ALTER TABLE bank.txn ADD COLUMN IF NOT EXISTS fraud_risk NUMERIC(5,1)")
r = run_sql("""INSERT INTO bank.txn
  (account_id, ts, amount, channel, anomaly_score, mcc, util_pct)
  VALUES ('CTA-100240', now(), 38500.00, 'SPEI', 8.7, NULL, 92.0)
  RETURNING txn_id, amount, anomaly_score""")
rid = r[0][0]
print("inserted txn_id", r[0])

## Step 2, compute a fraud-risk score and write it back

Compute a score from the row and write it back onto the same row: an analytics-derived value
served at operational latency. A high anomaly score and a near-maxed utilization push the
fraud risk up.

```sql
UPDATE bank.txn
SET fraud_risk = ROUND(100 * (anomaly_score / 10.0 + util_pct / 100.0), 1)
WHERE txn_id = <rid>;
```

In [ ]:
run_sql(f"UPDATE bank.txn SET fraud_risk = "
        f"ROUND(100*(anomaly_score/10.0 + util_pct/100.0),1) WHERE txn_id={rid}")
row = run_sql(f"SELECT txn_id, account_id, amount, anomaly_score, util_pct, fraud_risk "
              f"FROM bank.txn WHERE txn_id={rid}")[0]
print("scored + wrote back:", row)

## Step 3, the lakehouse side (same copy, no pipeline), EXPLAINER

This step is not run here: it needs the Lakebase database registered as a Unity Catalog online
catalog, which is a metastore-admin action (CREATE CATALOG). Once it exists, the SAME rows are
visible to a SQL warehouse with no pipeline and no synced copy. The query you would run:

```sql
-- on a SQL warehouse, against the UC foreign catalog for this database:
SELECT account_id, ts, amount, anomaly_score, fraud_risk
FROM <catalog>.bank.txn
WHERE account_id = 'CTA-100240' ORDER BY ts DESC LIMIT 1;
```

The transaction you wrote in Step 1 appears there immediately, because it is the same copy.
Nothing above is simulated: the write and the score are real on your branch; only this lakehouse
read needs the admin catalog setup, so we show the query honestly instead of faking its output.

## What you learned

One copy of the data served both the operational write and the analytics-derived fraud score, with
no pipeline between them. In a full setup the lakehouse reads the same rows through a UC foreign
catalog. Next: **Fits your landscape**, pointing standard Postgres tools at Lakebase.